In [ ]:
import papermill as pm
import pandas as pd
from multiprocessing import Pool
import concurrent.futures
import queue
import os
from pathlib import Path
from datetime import datetime

In [ ]:
def run_notebook(parameters,notebook_to_run,parameters_common={}):
    
    # change to directory where the notebook is (resolve relative imports)
    os.chdir(Path(notebook_to_run).absolute().parent)
    
    # run notebook
    for parameters_spec in parameters_list:
        parameters = {**parameters_common, **parameters_spec}

        pm.execute_notebook(
           notebook_to_run,
#             "/home/stumberger/image-analyis-recipes/alignment/correct_chromatic_aberration_for_tables_gs_copy.ipynb",
           '/dev/null',
           parameters=parameters)

# 0) Create projections

In [ ]:
parameters_list = [
    {"in_path": "/data/agl_data/NanoFISH/Gabi/GS198_Dppa3_all_mESC/"},
    {"in_path": "/data/agl_data/NanoFISH/Gabi/GS199_Dppa3_all_mESC/"}
]

notebook_to_run = "/home/stumberger/image-analyis-recipes/visualization/msr_make_projections_2_color.ipynb"

run_notebook(parameters_list,notebook_to_run)

# 1) Split channels and resave `msr` as `tif`

In [ ]:
parameters_list = [
    {"in_path": "/data/agl_data/NanoFISH/Gabi/GS198_Dppa3_all_mESC/"},
    {"in_path": "/data/agl_data/NanoFISH/Gabi/GS199_Dppa3_all_mESC/"}

]

notebook_to_run = "/home/stumberger/image-analyis-recipes/resave/resave_msr_as_tiff.ipynb"

run_notebook(parameters_list,notebook_to_run)

# 2) Spot detection

In [ ]:
parameters_list = [
    {"in_path": "/data/agl_data/NanoFISH/Gabi/GS198_Dppa3_all_mESC/"},
    {"in_path": "/data/agl_data/NanoFISH/Gabi/GS199_Dppa3_all_mESC/"}
]

parameters_common = {"channels": [0,1]}

notebook_to_run = "/home/stumberger/image-analyis-recipes/spot-detection/RS-FISH_spot_detection.ipynb"

run_notebook(parameters_list,notebook_to_run,parameters_common)

# 3) (optional) Join all spots into a big csv

In [ ]:
wd = "/data/agl_data/NanoFISH/Gabi/"

csv_files = [
#     "/data/agl_data/NanoFISH/Gabi/GS082_Dppa3_all-enhancers/20230926_run0_sted/naive/detections/",
#     "/data/agl_data/NanoFISH/Gabi/GS082_Dppa3_all-enhancers/20231003_run1_sted/naive/detections/",
]

# Initialize an empty list to store DataFrames
dataframes = []

# Read and store each CSV file as a DataFrame
for file in csv_files:
    df = pd.read_csv(f"{file}/merge.csv")
    dataframes.append(df)

# Join the DataFrames using Pandas (e.g., concatenate them vertically)
joined_dataframe = pd.concat(dataframes, ignore_index=True)

# Save the joined DataFrame to a new CSV file
time =  datetime.now().strftime("%Y-%m-%d-%H-%M")
joined_dataframe.to_csv(f'{wd}/sted_distances_{time}.csv', index=False)